In [ ]:
%pip install mne hu-neuro-pipeline openneuro-py

from mne import set_bipolar_reference
from mne.io import read_raw
from mne.preprocessing import ICA
from pipeline.datasets import get_erpcore

In [ ]:
from pathlib import Path
import openneuro

# Start Jupyter in the repository folder.
DATA_DIR = Path("ds003061").resolve()

def download_subject_data(dataset_id="ds003061", subject_id="sub-001", target_dir=DATA_DIR):
    """Download or resume this subject's files, even if the folder exists."""
    target_dir = Path(target_dir).expanduser().resolve()
    print(f"Checking/downloading {subject_id} from {dataset_id} into {target_dir}")
    openneuro.download(
        dataset=dataset_id,
        target_dir=str(target_dir),
        include=[f"{subject_id}/**"],
    )
    return target_dir


In [ ]:
from mne.io import read_raw_eeglab

# ds003061 contains EEGLAB .set files, not the ERP CORE .vhdr filename.
eeg_file = DATA_DIR / "sub-001" / "eeg" / "sub-001_task-P300_run-1_eeg.set"
if not eeg_file.is_file():
    download_subject_data()
if not eeg_file.is_file():
    raise FileNotFoundError(f"Download did not produce the EEG file: {eeg_file}")

print(f"Loading: {eeg_file}")
raw = read_raw_eeglab(eeg_file, preload=True)
data = raw.get_data()  # NumPy array: channels x samples, in volts.
print(f"Data shape: {data.shape}; sampling rate: {raw.info['sfreq']} Hz")
raw


In [ ]:
raw.get_data()


In [ ]:
raw.get_data().shape

In [ ]:
raw.plot(start=60, duration=10)

In [ ]:
raw.set_channel_types({"Temp": "misc"})
raw.set_montage("biosemi64", match_case=False)
raw.plot_sensors(show_names=True)

In [ ]:
raw = raw.set_eeg_reference('average')
raw.plot(start=60, duration=10)


In [ ]:
raw.filter(l_freq=1, h_freq=None)

raw.plot(start=60, duration=10)

In [ ]:
raw_copy = raw.copy()
ica = ICA(n_components=20)
ica.fit(raw_copy)
ica.plot_components()
ica.plot_sources(raw_copy)

In [ ]:
ica.plot_properties(raw_copy, picks=[0, 3, 6, 12])

In [ ]:
ica.plot_overlay(
    raw_copy, exclude=[0, 6],
    start=0.0, stop=20.0
)

In [ ]:
_ = ica.plot_overlay(
    raw_copy, exclude=[0, 6],
    picks=["Fp1", "Fp2", "AF7", "AF8"],
    start=0.0, stop=20.0
)


In [ ]:
ica.exclude = [0, 6]
raw_clean = ica.apply(raw.copy())
raw_clean.plot(start=60, duration=10)

In [ ]:
raw_clean.compute_psd(
    method="welch", fmin=1, fmax=100,
    picks=["FT7", "C5", "T7", "Cz", "Pz"]
).plot(average=False)

In [ ]:
ica.plot_properties(raw_copy, picks=[4])

In [ ]:
_ = ica.plot_overlay(
    raw, exclude=[0, 4, 6],
    picks=["FT7", "C5", "T7", "Cz"],
    start=60.0, stop=80.0
)

In [ ]:
ica.exclude = [0, 4, 6]
raw_clean = ica.apply(raw.copy())

In [ ]:
_ = ica.plot_overlay(
    raw, exclude=[0, 4, 6], picks=["Cz"],
    start=60.0, stop=80.0
)

In [ ]:
# Save the cleaned recording for analyisis.ipynb.
# Recheck ICA exclusions above after fitting a new decomposition.
clean_file = DATA_DIR.parent / "sub-001_run-1_clean_raw.fif"
raw_clean.save(clean_file, overwrite=True)
print(f"Saved: {clean_file}")